# Genex Therapy Agents — All Categories, Live Intake, and Conservative Support Logic

This notebook is a cleaned multi-category version of your earlier gross-motor-only notebook.

## Design note
**intake agent → delay agent → milestone Q&A agent → support-plan agent → summary agent**

The run path is intentionally direct and reliable for notebook use.

In [1]:
# ------------------------------------------------------------
# 0. Install once in the active notebook kernel if needed
# ------------------------------------------------------------
# %pip install pandas openpyxl openai python-dotenv ipython

In [11]:
from pathlib import Path
from datetime import datetime

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "therapy_agents"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

In [2]:
# ------------------------------------------------------------
# 1. Imports, setup, and CDC loading
# ------------------------------------------------------------
import os
import re
import json
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display, Markdown

try:
    from openai import OpenAI
except Exception:
    OpenAI = None

load_dotenv()

if OpenAI is None:
    print("Warning: openai package is not available yet.")
if not os.environ.get("OPENAI_API_KEY"):
    print("Warning: OPENAI_API_KEY is not visible. LLM-backed steps will fail until the key is set.")

DOMAIN_CONFIG = {
    "movement_and_physical": {
        "display": "Movement / Physical",
        "short": "motor",
    },
    "social_and_emotial": {
        "display": "Social / Emotional",
        "short": "social_emotional",
    },
    "language_and_communication": {
        "display": "Language / Communication",
        "short": "language_communication",
    },
    "cognitive": {
        "display": "Cognitive / Adaptive",
        "short": "cognitive",
    },
}

# Note: the uploaded CDC file uses "social and emotial" spelling.
ALIAS_TO_CATEGORY = {
    "movement and physical": "movement_and_physical",
    "physical": "movement_and_physical",
    "motor": "movement_and_physical",
    "gross motor": "movement_and_physical",
    "social and emotial": "social_and_emotial",
    "social and emotional": "social_and_emotial",
    "social_emotional": "social_and_emotial",
    "social": "social_and_emotial",
    "language and communication": "language_and_communication",
    "language": "language_and_communication",
    "speech": "language_and_communication",
    "speech and language": "language_and_communication",
    "cognitive": "cognitive",
    "adaptive": "cognitive",
}

ANSWER_SCORES = {
    "yes": 1.0,
    "sometimes": 0.45,
    "with_help": 0.2,
    "no": 0.0,
    "not_sure": 0.1,
}

VALID_ANSWERS = set(ANSWER_SCORES.keys())

def find_cdc_file(filename="milestone-cdc-table.xlsx"):
    search_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for root in search_roots:
        candidate = root / filename
        if candidate.exists():
            return candidate.resolve()
    for root in search_roots:
        if root.exists():
            matches = list(root.rglob(filename))
            if matches:
                return matches[0].resolve()
    raise FileNotFoundError(
        f"Could not find {filename}. Put it in the repo root or notebooks parent folder."
    )

def load_cdc_table(path=None):
    path = Path(path) if path else find_cdc_file()
    df = pd.read_excel(path)
    df.columns = [str(c).strip().lower() for c in df.columns]
    df = df.rename(columns={"category ": "category", "milestone ": "milestone"})
    df["category"] = df["category"].astype(str).str.strip().str.lower()
    df["milestone"] = df["milestone"].astype(str).str.strip()
    df["months"] = pd.to_numeric(df["months"], errors="coerce")
    df = df.dropna(subset=["months", "category", "milestone"]).copy()
    df["months"] = df["months"].astype(int)
    df["category_key"] = df["category"].map(lambda x: ALIAS_TO_CATEGORY.get(x, x.replace(" ", "_")))
    df["question_id"] = [f"{row.category_key}_{row.months}_{i}" for i, row in enumerate(df.itertuples(), start=1)]
    return df, path

cdc_df, cdc_path = load_cdc_table()
CDC_AGES = sorted(cdc_df["months"].dropna().unique().tolist())
print("Loaded CDC file:", cdc_path)
print("CDC ages:", CDC_AGES)
print("Rows:", len(cdc_df))

def get_category_questions(category_key: str, min_months: int, max_months: int) -> pd.DataFrame:
    subset = cdc_df[
        (cdc_df["category_key"] == category_key)
        & (cdc_df["months"] >= min_months)
        & (cdc_df["months"] <= max_months)
    ].sort_values(["months", "milestone"])
    return subset.copy()

def normalize_answer(answer_text: str) -> str:
    t = str(answer_text).strip().lower().replace(" ", "_")
    if t in VALID_ANSWERS:
        return t
    if t in {"notsure", "unsure", "maybe"}:
        return "not_sure"
    return "not_sure"

def print_banner(title: str):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

Loaded CDC file: /Users/sara/Projects/Genex/milestone-cdc-table.xlsx
CDC ages: [2, 4, 6, 9, 12, 15, 18, 24, 30, 36, 48, 60]
Rows: 159


In [4]:
# ------------------------------------------------------------
# 2. State + "agents"
# ------------------------------------------------------------

def new_state() -> Dict[str, Any]:
    return {
        "child": {},
        "delay_estimates": {},
        "qna": {},
        "dev_age": {},
        "support_recommendations": {},
    }

def intake_agent_live(state: Dict[str, Any]) -> Dict[str, Any]:
    print_banner("INTAKE AGENT")
    name = input("What is your child's name? ").strip()
    age_years = float(input("How old is your child in years? ").strip())
    diagnosis = input("Any diagnosis / condition? (type 'none' if none) ").strip()
    concern = input("What are your main concerns right now? ").strip()

    state["child"] = {
        "name": name,
        "age_years": age_years,
        "chronological_months": int(round(age_years * 12)),
        "diagnosis": diagnosis,
        "concern": concern,
    }

    return state["child"]

def estimate_delay_gpt(
    diagnosis: str,
    concern: str,
    chronological_months: int,
    category_key: str,
    model: str = "gpt-4o-mini",
) -> Dict[str, Any]:
    category_display = DOMAIN_CONFIG[category_key]["display"]

    if OpenAI is None or not os.environ.get("OPENAI_API_KEY"):
        # conservative fallback for when API is not available
        return {
            "delay_months": 12,
            "reason": f"Fallback delay estimate used for {category_display} because the OpenAI client is not available.",
            "source": "fallback",
        }

    client = OpenAI()
    prompt = f"""
You are helping with a NON-DIAGNOSTIC pediatric developmental intake.

Child information:
- Chronological age in months: {chronological_months}
- Diagnosis / condition: {diagnosis}
- Parent concern: {concern}
- Domain to estimate: {category_display}

Task:
Estimate a reasonable STARTING developmental delay in months for this single domain only.
This is only a rough anchor for question selection, not a diagnosis.

Rules:
- Be conservative.
- If there is little reason to think this domain is affected, you may return 0 to 6 months.
- If clearly affected, return a larger number.
- Never exceed the child's chronological age.
- Return strict JSON only.

Required JSON:
{{
  "delay_months": <integer>,
  "reason": "<one short sentence>"
}}
""".strip()

    try:
        resp = client.chat.completions.create(
            model=model,
            temperature=0.2,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": "You return strict JSON only."},
                {"role": "user", "content": prompt},
            ],
        )
        parsed = json.loads(resp.choices[0].message.content)
        delay_months = int(max(0, min(int(parsed.get("delay_months", 12)), chronological_months)))
        return {
            "delay_months": delay_months,
            "reason": parsed.get("reason", ""),
            "source": "openai",
        }
    except Exception as e:
        return {
            "delay_months": 12,
            "reason": f"Fallback delay estimate used because the OpenAI call failed: {e}",
            "source": "fallback",
        }

def delay_agent_all_categories(state: Dict[str, Any], categories: Optional[List[str]] = None) -> Dict[str, Any]:
    print_banner("DELAY ESTIMATOR AGENT")
    if not state.get("child"):
        raise ValueError("Child profile missing. Run intake first.")

    categories = categories or list(DOMAIN_CONFIG.keys())
    child = state["child"]

    for category_key in categories:
        est = estimate_delay_gpt(
            diagnosis=child["diagnosis"],
            concern=child["concern"],
            chronological_months=child["chronological_months"],
            category_key=category_key,
        )
        state["delay_estimates"][category_key] = est
        print(f"{DOMAIN_CONFIG[category_key]['display']}: {est['delay_months']} month starting delay | {est['reason']}")

    return state["delay_estimates"]

def build_milestone_questions(state: Dict[str, Any], category_key: str, window_months: int = 12, max_questions: int = 8) -> List[Dict[str, Any]]:
    child = state["child"]
    chrono_months = child["chronological_months"]
    delay_months = state["delay_estimates"].get(category_key, {}).get("delay_months", 12)

    approx_dev_months = max(2, chrono_months - delay_months)
    min_months = max(2, approx_dev_months - window_months // 2)
    max_months = min(60, approx_dev_months + window_months // 2)

    subset = get_category_questions(category_key, min_months=min_months, max_months=max_months)
    if subset.empty:
        subset = get_category_questions(category_key, min_months=min(CDC_AGES), max_months=max(CDC_AGES))

    questions = []
    for _, row in subset.iterrows():
        questions.append({
            "question_id": row["question_id"],
            "months": int(row["months"]),
            "milestone": row["milestone"],
            "question_text": f"Can {child['name']} {row['milestone']} right now? (yes / sometimes / with_help / no / not_sure)",
        })

    # Sort by month and keep a manageable window
    questions = sorted(questions, key=lambda x: (x["months"], x["milestone"]))
    return questions[:max_questions]

def compute_dev_age_from_answers(answers: List[Dict[str, Any]]) -> int:
    """
    Conservative weighted heuristic:
    - strong pass: yes
    - partial pass: sometimes / with_help / not_sure
    - developmental age = highest strong pass;
      if none, use one band lower than the earliest partial;
      if only no answers, use minimum asked band
    """
    if not answers:
        return 6

    answered_months = sorted(set([a["months"] for a in answers]))
    yes_months = [a["months"] for a in answers if a["norm_answer"] == "yes"]
    partial_months = [a["months"] for a in answers if a["norm_answer"] in {"sometimes", "with_help", "not_sure"}]

    if yes_months:
        return int(max(yes_months))

    if partial_months:
        anchor = int(min(partial_months))
        lower_candidates = [m for m in answered_months if m < anchor]
        return int(max(lower_candidates)) if lower_candidates else int(anchor)

    return int(min(answered_months))

def qna_agent_live(state: Dict[str, Any], category_key: str, ask_limit: int = 5) -> Dict[str, Any]:
    print_banner(f"MILESTONE Q&A AGENT — {DOMAIN_CONFIG[category_key]['display']}")
    questions = build_milestone_questions(state, category_key)
    asked = []

    for q in questions[:ask_limit]:
        ans = input(q["question_text"] + "\n").strip()
        norm = normalize_answer(ans)
        asked.append({
            "question_id": q["question_id"],
            "months": q["months"],
            "milestone": q["milestone"],
            "raw_answer": ans,
            "norm_answer": norm,
            "score": ANSWER_SCORES[norm],
        })

    dev_age = compute_dev_age_from_answers(asked)
    state["qna"][category_key] = asked
    state["dev_age"][category_key] = dev_age

    chrono = state["child"]["chronological_months"]
    print(f"Estimated developmental age for {DOMAIN_CONFIG[category_key]['display']}: {dev_age} months (chronological age {chrono} months)")
    return {
        "category": category_key,
        "dev_age_months": dev_age,
        "answers": asked,
    }

def no_special_support_needed(state: Dict[str, Any], category_key: str) -> bool:
    child = state["child"]
    chrono = min(child["chronological_months"], 60)
    dev_age = state["dev_age"].get(category_key, chrono)
    delay_est = state["delay_estimates"].get(category_key, {}).get("delay_months", 0)

    # If estimated delay is almost none, do not generate a therapy plan.
    return (chrono - dev_age) <= 6 or delay_est <= 6

def select_next_milestones(state: Dict[str, Any], category_key: str, max_milestones: int = 6) -> Dict[str, Any]:
    child = state["child"]
    dev_age = state["dev_age"].get(category_key, None)
    if dev_age is None:
        raise ValueError(f"No developmental age found for {category_key}. Run Q&A first.")

    if no_special_support_needed(state, category_key):
        return {
            "status": "no_special_support",
            "message": f"We do not think {child['name']} has a meaningful delay in {DOMAIN_CONFIG[category_key]['display']} and may not need special support in this category right now.",
            "milestones": [],
        }

    min_m = dev_age + 1
    max_m = min(60, dev_age + 12)
    subset = get_category_questions(category_key, min_months=min_m, max_months=max_m)

    milestones = []
    for _, row in subset.iterrows():
        milestones.append({
            "months": int(row["months"]),
            "milestone": row["milestone"],
        })

    return {
        "status": "success",
        "milestones": milestones[:max_milestones],
    }

def generate_support_plan(state: Dict[str, Any], category_key: str, model: str = "gpt-4o-mini") -> Dict[str, Any]:
    child = state["child"]
    category_display = DOMAIN_CONFIG[category_key]["display"]

    next_steps = select_next_milestones(state, category_key)
    if next_steps["status"] == "no_special_support":
        state["support_recommendations"][category_key] = {
            "status": "no_special_support",
            "summary": next_steps["message"],
            "plan": None,
        }
        return state["support_recommendations"][category_key]

    milestone_lines = "\n".join([f"- ({m['months']} months) {m['milestone']}" for m in next_steps["milestones"]])
    dev_age = state["dev_age"][category_key]

    if OpenAI is None or not os.environ.get("OPENAI_API_KEY"):
        plan = {
            "day_1": [f"Simple home practice for {category_display.lower()}"],
            "day_2": [f"Repeat with slight variation for {category_display.lower()}"],
            "day_3": [f"Practice during a familiar routine for {category_display.lower()}"],
            "day_4": [f"Use play-based repetition for {category_display.lower()}"],
            "day_5": [f"Review progress and repeat the easiest activity for {category_display.lower()}"],
        }
        state["support_recommendations"][category_key] = {
            "status": "fallback",
            "summary": f"Fallback plan used for {category_display} because OpenAI was unavailable.",
            "plan": plan,
        }
        return state["support_recommendations"][category_key]

    client = OpenAI()
    prompt = f"""
You are a pediatric therapist helping a parent at home.

Child:
- Name: {child['name']}
- Chronological age: {child['age_years']} years ({child['chronological_months']} months)
- Diagnosis / condition: {child['diagnosis']}
- Parent concern: {child['concern']}
- Category: {category_display}
- Estimated developmental age in this category: {dev_age} months

Next-step milestones:
{milestone_lines}

Task:
Create a 5-day home support plan with 1 to 2 short activities per day.
Use parent-friendly, practical language.
Do not overclaim.
Do not say "therapy model" or use medical jargon.
Return strict JSON only.

Required JSON:
{{
  "day_1": ["...", "... optional"],
  "day_2": ["..."],
  "day_3": ["..."],
  "day_4": ["..."],
  "day_5": ["..."]
}}
""".strip()

    try:
        resp = client.chat.completions.create(
            model=model,
            temperature=0.3,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": "You return strict JSON only and stay non-diagnostic."},
                {"role": "user", "content": prompt},
            ],
        )
        plan = json.loads(resp.choices[0].message.content)
        state["support_recommendations"][category_key] = {
            "status": "success",
            "summary": f"Created a 5-day support plan for {category_display}.",
            "plan": plan,
        }
        return state["support_recommendations"][category_key]
    except Exception as e:
        plan = {
            "day_1": [f"Short home activity for {category_display.lower()}"],
            "day_2": [f"Repeat with slight variation for {category_display.lower()}"],
            "day_3": [f"Practice inside a familiar routine for {category_display.lower()}"],
            "day_4": [f"Play-based repetition for {category_display.lower()}"],
            "day_5": [f"Review progress for {category_display.lower()}"],
        }
        state["support_recommendations"][category_key] = {
            "status": "fallback",
            "summary": f"Fallback plan used for {category_display} because the OpenAI call failed: {e}",
            "plan": plan,
        }
        return state["support_recommendations"][category_key]

def summarizer_agent(state: Dict[str, Any]) -> pd.DataFrame:
    child = state["child"]
    rows = []
    for category_key in DOMAIN_CONFIG:
        reco = state["support_recommendations"].get(category_key, {})
        rows.append({
            "child_name": child.get("name"),
            "chronological_age_months": child.get("chronological_months"),
            "diagnosis": child.get("diagnosis"),
            "category": DOMAIN_CONFIG[category_key]["display"],
            "estimated_delay_months": state["delay_estimates"].get(category_key, {}).get("delay_months"),
            "estimated_dev_age_months": state["dev_age"].get(category_key),
            "needs_special_support": False if reco.get("status") == "no_special_support" else True,
            "summary": reco.get("summary", ""),
        })
    return pd.DataFrame(rows)

def run_all_categories_live():
    state = new_state()
    intake_agent_live(state)
    delay_agent_all_categories(state)

    for category_key in DOMAIN_CONFIG:
        qna_agent_live(state, category_key)
        generate_support_plan(state, category_key)

    summary_df = summarizer_agent(state)
    return state, summary_df

In [5]:
# Optional quick sanity check
display(cdc_df.head())

,months,category,milestone,category_key,question_id
0,2,social and emotial,calms down when spoken to or picked up,social_and_emotial,social_and_emotial_2_1
1,2,social and emotial,looks at your face,social_and_emotial,social_and_emotial_2_2
2,2,social and emotial,seems happy to see you when you walk up to her,social_and_emotial,social_and_emotial_2_3
3,2,social and emotial,smiles when you talk to or smile at her,social_and_emotial,social_and_emotial_2_4
4,2,language and communication,makes sounds other than crying,language_and_communication,language_and_communication_2_5


In [10]:
# Run a full live session across all 4 categories.
state, summary_df = run_all_categories_live()
summary_df


INTAKE AGENT

DELAY ESTIMATOR AGENT
Movement / Physical: 6 month starting delay | Hyperactivity may impact physical coordination and control.
Social / Emotional: 6 month starting delay | Concerns about hyperactivity and lack of concentration suggest some social/emotional challenges.
Language / Communication: 6 month starting delay | Concerns about hyperactivity and lack of concentration may impact language development.
Cognitive / Adaptive: 6 month starting delay | Concerns about hyperactivity and lack of concentration suggest potential cognitive challenges.

MILESTONE Q&A AGENT — Movement / Physical
Estimated developmental age for Movement / Physical: 60 months (chronological age 72 months)

MILESTONE Q&A AGENT — Social / Emotional
Estimated developmental age for Social / Emotional: 60 months (chronological age 72 months)

MILESTONE Q&A AGENT — Language / Communication
Estimated developmental age for Language / Communication: 60 months (chronological age 72 months)

MILESTONE Q&A AGE

,child_name,chronological_age_months,diagnosis,category,estimated_delay_months,estimated_dev_age_months,needs_special_support,summary
0,Emma,72,ADHD,Movement / Physical,6,60,False,We do not think Emma has a meaningful delay in...
1,Emma,72,ADHD,Social / Emotional,6,60,False,We do not think Emma has a meaningful delay in...
2,Emma,72,ADHD,Language / Communication,6,60,False,We do not think Emma has a meaningful delay in...
3,Emma,72,ADHD,Cognitive / Adaptive,6,60,False,We do not think Emma has a meaningful delay in...


In [7]:
# Example: inspect one category's answers
state["qna"]["language_and_communication"][:5] if "language_and_communication" in state["qna"] else "Not run yet."

[{'question_id': 'language_and_communication_30_106',
  'months': 30,
  'milestone': 'name things in a book when you point and ask what is this',
  'raw_answer': 'yes',
  'norm_answer': 'yes',
  'score': 1.0},
 {'question_id': 'language_and_communication_30_105',
  'months': 30,
  'milestone': 'say two or more words toghether with one action word like doggie run',
  'raw_answer': 'sometimes',
  'norm_answer': 'sometimes',
  'score': 0.45},
 {'question_id': 'language_and_communication_30_104',
  'months': 30,
  'milestone': 'says about 50 words',
  'raw_answer': 'no',
  'norm_answer': 'no',
  'score': 0.0},
 {'question_id': 'language_and_communication_30_107',
  'months': 30,
  'milestone': 'says words like I me or we',
  'raw_answer': 'no',
  'norm_answer': 'no',
  'score': 0.0},
 {'question_id': 'language_and_communication_36_119',
  'months': 36,
  'milestone': 'ask who or what or where or why questions like where is mommy or where is daddy',
  'raw_answer': 'yes',
  'norm_answer': '

In [8]:
# Example: inspect one category's support output
state["support_recommendations"]["language_and_communication"] if "language_and_communication" in state["support_recommendations"] else "Not run yet."

{'status': 'success',
 'summary': 'Created a 5-day support plan for Language / Communication.',
 'plan': {'day_1': ["Ask Ari simple questions about her toys, like 'What is this toy for?' and encourage her to answer in complete sentences.",
   'Sing a favorite nursery rhyme together and pause to let Ari fill in the missing words.'],
  'day_2': ['During snack time, ask Ari to tell you about her favorite food. Encourage her to use four or more words in her answer.',
   'Read a short story together and ask Ari to repeat a line or two from the story.'],
  'day_3': ['Play a game where you describe an object in the room, and Ari has to guess what it is. Encourage her to ask questions.',
   'After a fun activity, ask Ari to tell you one thing she enjoyed doing today. Prompt her to use a full sentence.'],
  'day_4': ['While drawing or coloring, ask Ari what she is drawing and encourage her to describe it in four or more words.',
   'Play a song and encourage Ari to sing along, focusing on repea

In [ ]:
# Export the summary
summary_df.to_csv(OUTPUT_DIR/f"genex_therapy_all_categories_summary_{RUN_STAMP}.csv", index=False)
print("Saved genex_therapy_all_categories_summary.csv")

Saved genex_therapy_all_categories_summary.csv
